# Gemma 4 Agent: local evaluation that matches the scorer (127/129 gold patches pass)

Before I trust any local score, I want to know the ceiling: if my agent wrote the **reference (gold) patch**, would the official grader pass it?

In a plain Kaggle notebook (`swegemma`, `sandbox='subprocess'`) the answer is **no for 77 of the 129 public tasks**. The gold patch passes on only 52 (fastapi: 10 of 67). So a local number from that setup says more about the notebook than about the agent.

In this notebook I rebuild the scorer's sandbox inside the notebook and get **127 of 129**. The last 2 public tasks cannot pass anywhere: their required tests contradict each other (details at the end).

| | rich | fastapi | requests | httpx | total |
|---|---|---|---|---|---|
| plain notebook | 39 / 48 | 10 / 67 | 3 / 13 | 0 / 1 | **52 / 129** |
| this notebook | 47 / 48 | 66 / 67 | 13 / 13 | 1 / 1 | **127 / 129** |

Everything runs on CPU, with internet off. The wheels and interpreters are in two datasets:
[Local Eval Wheels](https://www.kaggle.com/datasets/zbigniewkirchner/gemma-4-agent-local-eval-wheels) and
[Local Eval Pythons](https://www.kaggle.com/datasets/zbigniewkirchner/gemma-4-agent-local-eval-pythons).

The same hook also works for the agent phase, so you can use this setup to score a real agent locally.

## What was wrong

I read `swegemma/harness`, `sandbox/setup.py` and `docker/Dockerfile.sandbox` from the competition data. The scorer runs each task in a clean `python:3.13-slim` image with pytest 8.4.2, the newest version of every wheel from `wheels/`, two shims (`imp.py`, `telnetlib.py`), and the repo itself on the path. In a notebook, none of that happens:

1. **The sandbox venv sees every Kaggle package**: IPython adds Pygments lexers, langsmith loads itself as a pytest plugin, and the versions are different.
2. **`install_test_dependencies` returns immediately** for `sandbox='subprocess'`, so nothing from `wheels/` is installed.
3. **Test-only packages are missing** from `wheels/`: `inline_snapshot`, `dirty_equals`, `pytest-httpbin`, `annotated-doc`, `shellingham` and 50 more.

So I give every sandbox a clean venv (no system packages) that points to a "scorer image" folder built once per repo. On top of that comes a small, explicit **calibration** for the public tasks. The scorer does not do it, but these tasks were not filtered the way the hidden ones were: their tests expect the package versions or the Python of their own time.

In [ ]:
import asyncio, glob, importlib, json, logging, os, re, shutil, subprocess, sys, tarfile, time, tomllib, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

COMP = Path(glob.glob('/kaggle/input/**/tasks.jsonl', recursive=True)[0]).parent
WHEELHOUSE = Path(glob.glob('/kaggle/input/**/adk_submission-*.whl', recursive=True)[0]).parent
EXTRA = Path(glob.glob('/kaggle/input/**/inline_snapshot-*.whl', recursive=True)[0]).parent
PYDS = Path(glob.glob('/kaggle/input/**/cpython-3.12-linux.tgz.bin', recursive=True)[0]).parent
assert (COMP / 'docker' / 'imp.py').exists()
print(COMP, WHEELHOUSE, EXTRA, PYDS, sep='\n')

I install the harness from the wheelhouse without the big CUDA wheels (no vLLM, no GPU needed). This image runs Python 3.13, and the wheelhouse also has cp312 builds, so I filter by tag.

In [ ]:
def fits(w, tag='cp313'):
    py, abi, plat = w.name[:-4].split('-')[-3:]
    if plat != 'any' and 'linux' not in plat:
        return False
    if abi == 'abi3':   # cp311-abi3 works on 3.11 and newer
        return int(py.split('.')[0][2:]) <= int(tag[2:])
    return any(t.startswith('py3') for t in py.split('.')) or tag in py.split('.')


tmp = Path('/tmp/wheelhouse'); tmp.mkdir(exist_ok=True)
for w in WHEELHOUSE.glob('*.whl'):   # Kaggle strips '+cu128' from wheel names
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp / name).exists():
        os.symlink(w, tmp / name)
small = [str(w) for w in tmp.glob('*.whl') if w.stat().st_size < 50 * 1024**2 and 'cutlass' not in w.name.lower() and fits(w)]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *small])
importlib.invalidate_caches()

import pandas as pd
from packaging.requirements import Requirement
from packaging.version import Version
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator
from swegemma.models import load_tasks
from swegemma.sandbox.subprocess import SubprocessManager
from adk_eval_core.tracing import SessionTrace
import swegemma.harness.verification as verification
import swegemma.harness.agent_runner as agent_runner

logging.getLogger().setLevel(logging.ERROR)
tasks = load_tasks(COMP / 'tasks.jsonl')
print(len(tasks), 'tasks')

This is the whole trick for checking gold patches: an `Evaluator` whose agent phase returns the reference patch.

In [ ]:
class GoldEvaluator(Evaluator):
    async def _run_agent_sandbox(self, task, snapshot_path, task_index, total_tasks, **kwargs):
        return task.patch, None, SessionTrace()


CURRENT = {'id': None}   # the task being graded


def run_all(tag, clean_venv):
    cfg = EvalConfig(tasks_path=COMP / 'tasks.jsonl', snapshots_dir=COMP / 'snapshots',
                     results_dir=Path(f'/kaggle/working/results_{tag}'), submission_dir=Path('/kaggle/working'),
                     models=None, sandbox='subprocess', wheels_dir=COMP / 'wheels', graph_dir=str(COMP / 'graphs'),
                     embeddings_dir=str(COMP / 'embeddings'), timeout_seconds=300, verification_timeout_seconds=300,
                     display_mode='quiet')
    ev = GoldEvaluator(cfg)
    if clean_venv:   # no Kaggle packages inside the sandbox
        ev.sandbox = ev.docker = SubprocessManager(timeout_seconds=300, system_site_packages=False)
    pool = ThreadPoolExecutor(max_workers=1)   # Jupyter already runs an event loop
    rows, t0 = [], time.time()
    for i, task in enumerate(tasks, 1):
        CURRENT['id'] = task.instance_id
        try:
            res = pool.submit(asyncio.run, ev.evaluate_task(task=task)).result()
            rows.append(dict(instance_id=task.instance_id, repo=task.repo, resolved=bool(res.resolved),
                             exit_code=res.test_exit_code, error=(res.error or '')[:200]))
        except Exception as e:
            rows.append(dict(instance_id=task.instance_id, repo=task.repo, resolved=False, exit_code=-1, error=repr(e)[:200]))
        if i % 20 == 0:
            print(f'[{tag}] {i}/{len(tasks)} resolved={sum(r["resolved"] for r in rows)} ({time.time() - t0:.0f}s)', flush=True)
    return pd.DataFrame(rows)

## Run 1: a plain notebook

In [ ]:
before = run_all('before', clean_venv=False)
print(before.groupby('repo').resolved.agg(['sum', 'count']), '\ntotal', before.resolved.sum())

## Run 2: a scorer-like sandbox

**Step 1, the scorer image.** For every repo I build one folder with the newest competition wheel of every package, pytest 8.4.2, pytest-timeout 2.1.0, the two shims and the missing test packages. I leave out the repo's own package: on the scorer, `pip install -e /workspace` replaces it with the code under test.

A few test-only packages go only to the repo that needs them. For example, with `brotlicffi` installed, httpx adds `br` to `Accept-Encoding`, and fastapi tests that compare request headers then fail.

In [ ]:
norm = lambda n: re.sub(r'[-_.]+', '_', n).lower()
WHEELS = {}   # package -> [(version, wheel)]: competition wheels + my test deps + cp312 builds
for w in [*(COMP / 'wheels').glob('*.whl'), *EXTRA.glob('*.whl'), *PYDS.glob('*.whl')]:
    name, ver = w.name.split('-')[:2]
    WHEELS.setdefault(norm(name), []).append((Version(ver), w))

OWN = {'fastapi/fastapi': 'fastapi', 'Textualize/rich': 'rich', 'psf/requests': 'requests', 'encode/httpx': 'httpx'}
ONLY_FOR = {'pytest_httpbin': {'psf/requests'}, 'httpbin': {'psf/requests'}, 'flasgger': {'psf/requests'},
            'brotlicffi': {'psf/requests', 'encode/httpx'}, 'brotli': {'encode/httpx'}, 'zstandard': {'encode/httpx'}}
PY = {'cp313': Path(sys.executable)}


def python_for(tag):   # cp313 = this notebook, cp312 = standalone build from the dataset
    if tag not in PY:
        ver = f'{tag[2]}.{tag[3:]}'
        dest = Path(f'/kaggle/working/py{tag}')
        if not (dest / 'python' / 'bin' / f'python{ver}').exists():
            with tarfile.open(PYDS / f'cpython-{ver}-linux.tgz.bin', 'r:gz') as t:   # renamed so Kaggle keeps it packed
                t.extractall(dest)
        PY[tag] = dest / 'python' / 'bin' / f'python{ver}'
    return PY[tag]


BASE = {}


def base_for(repo, tag):
    if (repo, tag) not in BASE:
        sel = {}
        for name, opts in WHEELS.items():
            opts = [(v, w) for v, w in opts if fits(w, tag)]
            if not opts or name == OWN[repo] or (name in ONLY_FOR and repo not in ONLY_FOR[name]):
                continue
            comp = [(v, w) for v, w in opts if w.parent == COMP / 'wheels']
            if name in ('pytest', 'pytest_timeout'):   # what Dockerfile.sandbox pip-installs
                sel[name] = next(w for v, w in opts if str(v) in ('8.4.2', '2.1.0'))
            elif comp:
                sel[name] = max(comp)[1]   # the scorer injects the newest competition wheel
            elif name != 'pygments':      # (old Pygments wheels are only for per-task pins)
                sel[name] = max(opts)[1]
        d = Path(f'/kaggle/working/scorer_site_{OWN[repo]}_{tag}')
        shutil.rmtree(d, ignore_errors=True)
        subprocess.run([str(python_for(tag)), '-m', 'pip', 'install', '-q', '--no-index', '--no-deps',
                        '--ignore-requires-python', '--target', str(d), *map(str, sel.values())])
        for shim in ('imp.py', 'telnetlib.py'):   # the same shims as the scorer image, from the competition's docker/ folder
            shutil.copy(COMP / 'docker' / shim, d / shim)
        BASE[(repo, tag)] = (d, sel)
    return BASE[(repo, tag)]

**Step 2, calibration for the public tasks.** I found every entry below by reading the pytest output of a failing gold patch. I wrote the reason next to each one.

- For every task, I install the newest version that the snapshot's own `pyproject.toml` allows, plus the exact versions those wheels require (`pydantic` needs its matching `pydantic-core`). Without it, old fastapi commits meet starlette 1.6 and fail with `Router.__init__() got an unexpected keyword argument 'on_startup'`.
- A few tasks need more than that:

In [ ]:
CALIBRATION = {   # task -> exact versions
    # old rich tests compare colored output 1:1, so they need the Pygments of their time
    'rich_3064': {'pygments': '2.15.1'}, 'rich_3130': {'pygments': '2.15.1'}, 'rich_3296': {'pygments': '2.16.1'},
    'rich_3469': {'pygments': '2.18.0'}, 'rich_3782': {'pygments': '2.19.2'}, 'rich_4070': {'pygments': '2.19.2'},
    'rich_4079': {'pygments': '2.20.0'},
    # the httpbin test server (werkzeug 3.1) crashes on a redirect to port -1, werkzeug 2.3 does not
    'requests_6589': {'werkzeug': '2.3.8', 'flask': '2.3.3'}, 'requests_6592': {'werkzeug': '2.3.8', 'flask': '2.3.3'},
    'requests_6629': {'werkzeug': '2.3.8', 'flask': '2.3.3'},
}
TASK_PYTHON = {'requests_6757': 'cp312',   # Python 3.13 turned on VERIFY_X509_STRICT, the test certificates fail it
               'rich_3472': 'cp312'}       # Python 3.13 changed the AttributeError message the test compares
ASYNC_SHIM = {'fastapi_15745', 'fastapi_15763', 'fastapi_15785'}   # async tests, and the harness passes -p no:anyio
NO_CERT_TIME = {'requests_6757'}   # the test certificates have expired since the task was made

SHIM = Path('/kaggle/working/shims')
(SHIM / 'swe_async_shim-0.1.dist-info').mkdir(parents=True, exist_ok=True)
(SHIM / 'swe_async_shim-0.1.dist-info' / 'METADATA').write_text('Metadata-Version: 2.1\nName: swe-async-shim\nVersion: 0.1\n')
(SHIM / 'swe_async_shim-0.1.dist-info' / 'entry_points.txt').write_text('[pytest11]\nswe_async_shim = swe_async_shim\n')
(SHIM / 'swe_async_shim.py').write_text('''import asyncio, inspect
import pytest


@pytest.hookimpl(tryfirst=True)
def pytest_pyfunc_call(pyfuncitem):   # run async def tests with asyncio.run
    if inspect.iscoroutinefunction(pyfuncitem.obj):
        args = {a: pyfuncitem.funcargs[a] for a in pyfuncitem._fixtureinfo.argnames}
        asyncio.run(pyfuncitem.obj(**args))
        return True
''')
(SHIM / 'swe_no_cert_time.py').write_text('''import ssl

_new = ssl.SSLContext.__new__


def _new_no_time(cls, *args, **kwargs):   # verify certificates as on the day the task was made
    ctx = _new(cls, *args, **kwargs)
    ctx.verify_flags |= 0x200000   # X509_V_FLAG_NO_CHECK_TIME
    return ctx


ssl.SSLContext.__new__ = staticmethod(_new_no_time)
''')

**Step 3, the hook.** `install_test_dependencies` is the harness function that does nothing in a notebook. I replace it in both the verification and the agent module. The clean venv has no `pip` (Debian Python, no `ensurepip`), so I install from the notebook straight into its `site-packages`.

In [ ]:
def poetry_to_pep440(spec):
    spec = spec if isinstance(spec, str) else spec.get('version', '')
    m = re.fullmatch(r'\^(\d+)(\.\d+)?(\.\d+)?', spec.strip())
    return f'>={spec.strip()[1:]},<{int(m.group(1)) + 1}' if m else ('' if spec.strip() == '*' else spec)


PICKS = {}


def setup_env(docker, sandbox_id, repo='', **kwargs):
    tid = CURRENT['id']
    tag = TASK_PYTHON.get(tid, 'cp313')
    site = Path(docker.exec(sandbox_id, 'python -c "import site; print(site.getsitepackages()[0])"').stdout.strip())
    if tag != 'cp313':   # rebuild the sandbox venv on another interpreter
        venv_dir = site.parents[2]
        subprocess.run([str(python_for(tag)), '-m', 'venv', '--clear', '--without-pip', str(venv_dir)], check=True)
        site = next(venv_dir.glob('lib/python3*/site-packages'))
    ws = docker.exec(sandbox_id, 'pwd').stdout.strip()
    base, sel = base_for(repo, tag)
    # like the scorer: src/ layouts importable, the image packages, then /workspace subfolders (workspace_paths.pth)
    (site / '00_src.pth').write_text(f'{ws}/src\n')
    (site / '01_scorer_base.pth').write_text(f'{base}\n')
    (site / 'zz_workspace_paths.pth').write_text('\n'.join(str(p) for p in Path(ws).iterdir()
                                                           if p.is_dir() and not p.name.startswith('.')) + '\n')
    if tid in ASYNC_SHIM:
        (site / '02_async_shim.pth').write_text(f'{SHIM}\n')
    if tid in NO_CERT_TIME:   # a .pth line that starts with 'import' runs at interpreter startup
        (site / '03_no_cert_time.pth').write_text(f'import sys; sys.path.insert(0, {str(SHIM)!r}); import swe_no_cert_time\n')

    # versions the snapshot's pyproject.toml allows, and what those wheels require
    try:
        data = tomllib.loads(Path(ws, 'pyproject.toml').read_text())
    except Exception:
        data = {}
    reqs = list(data.get('project', {}).get('dependencies', []))
    reqs += [n + poetry_to_pep440(s) for n, s in data.get('tool', {}).get('poetry', {}).get('dependencies', {}).items() if n != 'python']
    picks, queue, seen = {}, list(reqs), set()
    while queue:
        try:
            req = Requirement(queue.pop(0))
        except Exception:
            continue
        key = norm(req.name)
        if key in seen or key == OWN.get(repo) or (req.marker and not req.marker.evaluate({'extra': ''})):
            continue
        seen.add(key)
        options = [(v, w) for v, w in WHEELS.get(key, []) if fits(w, tag) and req.specifier.contains(v, prereleases=True)
                   and not (key in ONLY_FOR and repo not in ONLY_FOR[key])]
        if options:
            best = max(options)[1]
            if best != sel.get(key):
                picks[key] = best
            with zipfile.ZipFile(best) as z:
                meta = next(n for n in z.namelist() if n.endswith('.dist-info/METADATA'))
                queue += [l.split(':', 1)[1].strip() for l in z.read(meta).decode().splitlines() if l.startswith('Requires-Dist:')]
    for name, ver in CALIBRATION.get(tid, {}).items():
        picks[name] = next(w for v, w in WHEELS[name] if str(v) == ver and fits(w, tag))
    if picks:
        subprocess.run([str(python_for(tag)), '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', '--upgrade',
                        '--ignore-requires-python', '--target', str(site), *map(str, picks.values())])
    PICKS[tid] = [tag] + sorted(p.name.split('-')[0] + '==' + p.name.split('-')[1] for p in picks.values())


verification.install_test_dependencies = setup_env   # verification sandbox
agent_runner.install_test_dependencies = setup_env   # agent sandbox, if you evaluate a real agent

In [ ]:
after = run_all('after', clean_venv=True)
print(after.groupby('repo').resolved.agg(['sum', 'count']), '\ntotal', after.resolved.sum())

## Result

In [ ]:
both = before[['instance_id', 'repo', 'resolved']].rename(columns={'resolved': 'gold_passes_plain_notebook'}).merge(
    after[['instance_id', 'resolved', 'exit_code', 'error']].rename(columns={'resolved': 'gold_passes_scorer_like'}), on='instance_id')
both['environment'] = both.instance_id.map(lambda t: ' '.join(PICKS.get(t, [])))
both['calibrated'] = both.instance_id.map(lambda t: t in CALIBRATION or t in TASK_PYTHON or t in ASYNC_SHIM)
both.to_csv('/kaggle/working/gold_patch_check.csv', index=False)

summary = both.groupby('repo')[['gold_passes_plain_notebook', 'gold_passes_scorer_like']].sum()
summary['tasks'] = both.groupby('repo').size()
summary.loc['total'] = summary.sum()
print(summary)
print('\nStill failing:')
print(both.loc[~both.gold_passes_scorer_like, ['instance_id', 'error']].to_string(index=False))

## The 2 tasks no environment can pass

The grader requires every test function from the hidden test patch to **pass, not be skipped**. Two public tasks require a pair of tests with opposite skip conditions:

- **rich_3486**: `test_traceback_finely_grained` has `skipif(sys.version_info.minor < 11)`, and `test_traceback_finely_grained_missing` has `skipif(sys.version_info.minor >= 11)`. On every Python, one of them is skipped.
- **fastapi_14186**: `test_model_field_default_required` is `@needs_pydanticv2` and `test_model_description_escaped_with_formfeed` is `@needs_pydanticv1`, and both check the same `PYDANTIC_V2` flag.

So **127 is the maximum** for the public set. The hidden tasks were filtered so that their gold patches pass on the scorer (organizers report 100%).

## How I would use this

- Validate an agent locally on the 127 tasks where `gold_passes_scorer_like` is true (`gold_patch_check.csv`, also in the dataset).
- Keep in mind that the calibrated tasks (the `calibrated` column) were found by trying versions with the gold patch. They make the public tasks gradable. They are not something your agent could know.